# Align before Fuse（ALBEF）模型

> **论文**：*Align before Fuse: Vision and Language Representation Learning with Momentum Distillation*（Li et al., NeurIPS 2021）  
> **核心创新**：先用 ITC 对比学习对齐单模态特征（Align），再用 Cross-Attention 融合（Fuse），避免了未对齐特征直接融合导致的次优解。  
> **三大训练任务**：ITC（图文对比）+ ITM（图文匹配）+ MLM（掩码语言模型）

## 一、ALBEF 基础架构（图像编码器 + 文本编码器 + 多模态融合 + 三任务）

本节手动搭建完整 ALBEF 模型，包含：
- `ImageEncoder`：ViT-B/16 骨干网络（与论文完全一致，输出 197 个 token，含 [CLS] + 196 个 patch token）
- `TextEncoder`：完整 12 层 BERT-base 文本编码器
- `MultimodalEncoder`：6 层 MultimodalEncoderLayer（自注意力 + 图文交叉注意力 + FFN）
- `ALBEF`：统一入口，根据 `task` 参数分发到 ITC / ITM / MLM 三个任务

### 1.1 导入依赖

导入 PyTorch、HuggingFace Transformers、torchvision 等所有后续代码依赖的第三方库。

In [9]:
# ============================================================
# ALBEF 教学版 Notebook - 依赖导入
# 内容：图像编码器 + 文本编码器 + 多模态融合 + 三任务（ITC/ITM/MLM）
# 论文要点：Align Before Fuse（先对比对齐，再 Cross-Attention 融合）
# ============================================================

import torch                              # PyTorch 核心库：张量计算、GPU 加速、自动求导
import torch.nn as nn                     # 神经网络模块：Linear、LayerNorm、Module 等
import torch.nn.functional as F           # 函数式 API：softmax、normalize、cross_entropy 等
from transformers import (
    BertModel,          # BERT 主体模型（12 层 Transformer，输出上下文向量）
    BertTokenizer,      # BERT 分词器（WordPiece，输出 input_ids / attention_mask）
    BertConfig,         # BERT 配置类（用于自定义层数、隐藏维度等超参数）
    ViTModel,           # ViT-B/16 主体模型（Patch Embedding + Transformer，输出 token 序列）
    ViTImageProcessor,  # ViT 配套图像预处理器（完成 resize → normalize → 转 pixel_values 张量）
)
from PIL import Image                      # Pillow 图像库：用于创建/加载 PIL.Image 对象，作为 ViTImageProcessor 的输入
import random                             # Python 随机数（演示时可复现）
import numpy as np                        # 数值计算库（设置随机种子、生成随机像素矩阵）
import math                               # 数学函数（注意力缩放因子 sqrt(d_k)）




### 1.2 图像编码器（ImageEncoder）

以 ViT-B/16 为骨干（与 ALBEF 论文一致），将输入图像切分为 14×14=196 个 patch，加上 [CLS] token 共输出 197 个 token 特征序列。[CLS] token（位于第 0 位）作为全局图像表示用于 ITC 对比学习，其余 196 个 patch token 作为 Key/Value 参与 Cross-Attention 融合。

In [10]:
class ImageEncoder(nn.Module):
    """
    图像编码器：将输入图像映射为 patch 级 token 特征序列（含 [CLS] token）。
    使用 ViT-B/16（Vision Transformer Base，patch_size=16），与 ALBEF 论文完全一致。

    主要属性：
        processor (ViTImageProcessor)：图像预处理器，与 ViT 权重同源加载，
            在 forward() 内部自动调用，完成 resize → normalize → 转 Tensor 三步预处理；
            调用方只需传入原始 PIL.Image 列表，无需手动归一化。
        vit (ViTModel)：预训练 ViT-B/16 骨干网络。
        proj (nn.Linear)：将 ViT 输出投影到与 BERT 对齐的统一语义维度。

    forward 输出形状：(batch_size, 197, embed_dim)
        其中 197 = 1 个 [CLS] token（位于第 0 位，全局图像语义，用于 ITC）
                  + 196 个 patch token（14×14 局部区域特征，用于 Cross-Attention 的 Key/Value）
    """

    def __init__(self, embed_dim=768, model_name="google/vit-base-patch16-224"):
        """
        初始化图像编码器。
        Args:
            embed_dim (int): 输出特征维度，需与 BERT 隐藏层维度一致，默认 768
            model_name (str): HuggingFace ViT 预训练模型名称，默认 ViT-B/16（224×224 输入）
        """
        super().__init__()           # 调用父类初始化，注册为 PyTorch 子模块

        # 加载与 ViT 模型配套的图像预处理器（ViTImageProcessor）
        # 功能：将原始 PIL.Image 或 numpy 数组转换为归一化的 pixel_values 张量
        # 具体步骤：① 将图像 resize 至 224×224；② 按 ImageNet 均值/标准差做像素归一化；③ 转为 torch.Tensor
        # 使用方式：forward() 接收已预处理的张量；若从原始图片出发，可先调用：
        #   inputs = encoder.processor(images=pil_images, return_tensors="pt")
        #   pixel_values = inputs["pixel_values"]   # 形状 (batch_size, 3, 224, 224)
        self.processor = ViTImageProcessor.from_pretrained(model_name, cache_dir="./model_cache/ViT")

        # 加载 HuggingFace 预训练 ViT-B/16 权重
        # 与 processor 共享同一 model_name，确保预处理参数（均值/方差）与模型训练时严格一致
        self.vit = ViTModel.from_pretrained(model_name, cache_dir="./model_cache/ViT")

        vit_hidden_size = self.vit.config.hidden_size  # 读取 ViT 的隐藏层维度（ViT-B/16 为 768）
        self.proj = nn.Linear(vit_hidden_size, embed_dim)  # 投影层：将 ViT 输出的 vit_hidden_size(768) 映射到统一语义维度 embed_dim(768)

    def forward(self, images):
        """
        前向传播：原始 PIL 图像列表 → patch token 特征序列（含 [CLS] token）。
        Args:
            images (List[PIL.Image.Image]): 长度为 batch_size 的 PIL 图像列表，
                                            尺寸任意（processor 内部统一 resize 至 224×224）
        Returns:
            features (Tensor): 形状 (batch_size, 197, embed_dim)；
                               197 = 14×14=196 个 patch token + 1 个 [CLS] token；
                               第 0 位为 [CLS] token，用作全局图像向量（ITC 对比学习使用）；
                               第 1~196 位为各 patch 局部视觉特征（Cross-Attention 中作为 Key/Value）
        """
        # 使用 ViTImageProcessor 完成三步预处理，将原始 PIL 图像转换为归一化张量：
        #   ① resize：将任意分辨率图像统一缩放至 224×224（ViT-B/16 要求的固定输入尺寸）
        #   ② normalize：按 ImageNet 均值 [0.5, 0.5, 0.5] 和标准差 [0.5, 0.5, 0.5] 做像素归一化
        #   ③ 转张量：输出 dict，键 "pixel_values" 对应 torch.Tensor，形状 (batch_size, 3, 224, 224)
        # 使用与 ViT 权重同源加载的 processor，确保归一化参数与预训练阶段严格一致
        inputs = self.processor(images=images, return_tensors="pt")

        # 将 pixel_values 搬到与模型参数相同的设备（CPU 或 GPU）
        # next(self.parameters()).device 自动读取当前模型所在的设备，无需硬编码
        # pixel_values 形状：(batch_size, 3, 224, 224)
        #   - batch_size：当前批次图像数量
        #   - 3          ：RGB 三个颜色通道
        #   - 224, 224   ：固定输入分辨率（16×16 的 patch 恰好切出 14×14=196 块）
        pixel_values = inputs["pixel_values"].to(next(self.parameters()).device)

        # ViT-B/16 主干前向传播：
        #   outputs 为 BaseModelOutputWithPooling 对象，主要字段：
        #     - last_hidden_state：最后一层 Transformer 的全部 token 输出（此处使用）
        #     - pooler_output    ：[CLS] token（last_hidden_state[:, 0]）经 Linear + 激活函数（默认 tanh，
        #                          由 ViTConfig.pooler_act 控制）后的汇聚向量；此处不使用，
        #                          因为我们需要完整的 197 个 token 序列而非单一全局向量
        outputs = self.vit(pixel_values=pixel_values)

        # 取最后一层隐藏状态，形状为 (batch_size, 197, vit_hidden_size)
        #   - batch_size     ：批次内图像数量
        #   - 197            ：token 总数 = 1 个 [CLS] token（位于第 0 位，代表全局图像语义）
        #                       + 196 个 patch token（14 行 × 14 列，代表各局部区域的视觉特征）
        #   - vit_hidden_size：ViT-B/16 的隐藏层维度，固定为 768
        features = outputs.last_hidden_state

        # 通过线性投影层将【全部 197 个 token】映射到与 BERT 对齐的统一语义空间
        # 输出形状：(batch_size, 197, embed_dim)
        #   - batch_size：批次内图像数量（不变）
        #   - 197        ：[CLS] token（第 0 位）+ 196 个 patch token（第 1~196 位），全部保留，各有用途：
        #
        #       ① ITC 对比学习（最核心需求）：
        #          严格要求 [CLS] token，即 features[:, 0, :]；
        #          ViT 的 [CLS] 经 self-attention 汇聚了全图语义，作为全局图像向量与文本 [CLS] 做相似度匹配。
        #          196 个 patch token 在 ITC 阶段不参与计算。
        #
        #       ② Cross-Attention（ITM / MLM 的 Fuse 阶段）：
        #          196 个 patch token 是核心 Key/Value，提供图像各局部区域的空间细节，
        #          文本 token 对其做注意力查询，定位"哪个词对应图像的哪个区域"；
        #          [CLS] token 也一并保留：它已通过 ViT self-attention 汇聚了全局视觉信息，
        #          作为额外的"全局摘要" Key/Value，使文本在关注局部细节的同时也能感知整体语义；
        #          理论上仅用 196 个 patch token 也可完成 Cross-Attention，
        #          但丢弃 [CLS] 会损失全局视觉锚点，实践中通常保留完整序列。
        #
        #   - embed_dim  ：统一嵌入维度（默认 768），与 BERT 隐藏层维度一致，
        #                  确保图文 token 特征可在同一向量空间做内积或 Cross-Attention
        features = self.proj(features)
        return features

### 1.3 文本编码器（TextEncoder）

基于 BERT-base 的文本单模态编码器，将 token ID 序列转化为上下文感知的稠密向量。

In [11]:
class TextEncoder(nn.Module):
    """
    文本编码器：使用完整 BERT-base 将 token 序列编码为上下文向量。
    对应 ALBEF 论文中文本单模态编码器（论文使用 BERT 前 6 层，此处用完整 12 层演示）。
    """

    def __init__(self, embed_dim=768):
        """
        初始化文本编码器。
        Args:
            embed_dim (int): BERT 隐藏层维度，默认 768
        """
        super().__init__()  # 调用父类初始化，注册为 PyTorch 子模块
        self.bert = BertModel.from_pretrained('bert-base-uncased',cache_dir="./model_cache/bert")  # 加载英文 BERT-base（12 层，768 维，12 头）
        self.embed_dim = embed_dim  # 记录嵌入维度，便于后续模块对齐

    def forward(self, input_ids, attention_mask):
        """
        前向传播：token ID 序列 → 上下文向量序列。
        Args:
            input_ids (Tensor): token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 1 表示有效 token，0 表示 padding，形状 (batch_size, seq_len)
        Returns:
            last_hidden_state (Tensor): 每个 token 的上下文向量，形状 (batch_size, seq_len, 768)；
                                        第 0 位 [CLS] 可作为句子级别的全局表示，用于 ITC 对比学习
        """
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)  # BERT 12 层 Transformer 前向传播
        return outputs.last_hidden_state  # 取最后一层（第 12 层）隐藏状态作为文本特征

### 1.4 多头交叉注意力（MultiHeadCrossAttention）

图文交叉注意力的核心实现：文本 token 作为 Query，图像 patch 作为 Key/Value，实现视觉信息向文本侧的注入。

In [12]:
class MultiHeadCrossAttention(nn.Module):
    """
    多头交叉注意力：Query 来自文本，Key/Value 来自图像。
    让文本 token「看到」图像中与之相关的区域，是 ALBEF Fuse 阶段的核心算子。
    """

    def __init__(self, embed_dim=768, num_heads=12):
        """
        初始化多头交叉注意力模块。
        Args:
            embed_dim (int): 输入输出维度，须能被 num_heads 整除，默认 768
            num_heads (int): 注意力头数，BERT-base 默认 12 头
        """
        super().__init__()                      # 调用父类初始化，注册多头交叉注意力子模块
        self.embed_dim = embed_dim              # 模型总维度
        self.num_heads = num_heads              # 多头数量
        self.head_dim = embed_dim // num_heads  # 每个头的维度 = 768/12 = 64
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"  # 断言维度可被头数整除，否则无法均匀分配注意力头
        self.q_linear = nn.Linear(embed_dim, embed_dim)  # Query 投影矩阵（来自文本侧）
        self.k_linear = nn.Linear(embed_dim, embed_dim)  # Key 投影矩阵（来自图像侧）
        self.v_linear = nn.Linear(embed_dim, embed_dim)  # Value 投影矩阵（来自图像侧）
        self.out = nn.Linear(embed_dim, embed_dim)       # 多头拼接后的输出投影

    def forward(self, query, key, value, attention_mask=None):
        """
        前向传播：计算文本对图像的多头交叉注意力。
        Args:
            query (Tensor): 文本特征，形状 (batch_size, text_len, embed_dim)
            key (Tensor): 图像特征，形状 (batch_size, img_patches, embed_dim)
            value (Tensor): 图像特征，形状 (batch_size, img_patches, embed_dim)
            attention_mask (Tensor, optional): 文本掩码，形状 (batch_size, text_len)
        Returns:
            Tensor: 交叉注意力输出，形状与 query 相同，(batch_size, text_len, embed_dim)
        """
        # query.size(0) 取 Tensor 第 0 维大小，即批次内的样本数量
        # query 形状：(batch_size, text_len, embed_dim)
        batch_size = query.size(0)

        # ── Q 的生成（来自文本）────────────────────────────────────────────
        # self.q_linear(query)
        #   线性投影文本特征，形状：(batch_size, text_len, embed_dim)
        #   - batch_size：批次内样本数量
        #   - text_len  ：文本序列长度（含 [CLS]/[SEP]/padding）
        #   - embed_dim ：768，投影后维度不变，但权重矩阵将特征映射到 Q 子空间
        # .view(batch_size, -1, self.num_heads, self.head_dim)
        #   将 embed_dim 拆分为 num_heads 个 head_dim，形状变为：
        #   (batch_size, text_len, num_heads=12, head_dim=64)
        # .transpose(1, 2)
        #   交换 text_len 与 num_heads 两个维度，使各头在第 1 维独立，形状变为：
        #   (batch_size, num_heads=12, text_len, head_dim=64)
        Q = self.q_linear(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # ── K 的生成（来自图像）────────────────────────────────────────────
        # self.k_linear(key)
        #   线性投影图像 token 特征，形状：(batch_size, img_patches, embed_dim)
        #   - img_patches：图像 token 数量，ViT-B/16 为 197（1 个 [CLS] + 196 个 patch token）
        # .view / .transpose 操作同 Q，形状最终变为：
        #   (batch_size, num_heads=12, img_patches=197, head_dim=64)
        K = self.k_linear(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # ── V 的生成（来自图像）────────────────────────────────────────────
        # self.v_linear(value)
        #   线性投影图像 token 特征，与 K 同源（key == value），形状：(batch_size, img_patches, embed_dim)
        # 形状最终变为：(batch_size, num_heads=12, img_patches=197, head_dim=64)
        # 注：K 决定"关注哪些图像 token"，V 决定"融合哪些图像信息"，分开投影使模型更灵活
        V = self.v_linear(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 调用缩放点积注意力：Attention(Q, K, V) = softmax(QK^T / sqrt(head_dim)) · V
        # 输入：Q (B, num_heads, text_len, head_dim)
        #        K (B, num_heads, img_patches, head_dim)
        #        V (B, num_heads, img_patches, head_dim)
        # 输出：attention (B, num_heads, text_len, head_dim)
        #   每个文本 token 对 197 个图像 token 做加权求和，融合视觉信息
        attention = self._attention(Q, K, V, attention_mask)

        # ── 多头拼接（concat heads）───────────────────────────────────────
        # .transpose(1, 2)
        #   将 num_heads 换回第 2 维，形状：(batch_size, text_len, num_heads=12, head_dim=64)
        # .contiguous()
        #   transpose 后内存不连续，contiguous() 重新分配连续内存，使后续 view() 不报错
        # .view(batch_size, -1, self.embed_dim)
        #   将 num_heads × head_dim = 12 × 64 = 768 合并为 embed_dim，形状还原为：
        #   (batch_size, text_len, embed_dim=768)
        #   这相当于将 12 个头的输出拼接（concat）为一个完整向量，与单头输出等价但表达能力更强
        attention = attention.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_dim)
        return self.out(attention)  # 输出线性变换，映射回 embed_dim

    def _attention(self, Q, K, V, attention_mask=None):
        """
        缩放点积注意力核心计算：Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) · V
        Args:
            Q (Tensor): Query，形状 (B, num_heads, text_len, head_dim)
            K (Tensor): Key，形状 (B, num_heads, img_patches, head_dim)
            V (Tensor): Value，形状 (B, num_heads, img_patches, head_dim)
            attention_mask (Tensor, optional): 掩码，形状 (B, text_len)
        Returns:
            Tensor: 加权输出，形状 (B, num_heads, text_len, head_dim)
        """
        # 计算缩放点积注意力分数 QK^T / sqrt(d_k)
        # Q：(B, num_heads, text_len, head_dim)
        # K^T：(B, num_heads, head_dim, img_patches)
        # scores：(B, num_heads, text_len, img_patches)
        #   scores[b, h, i, j] = 第 b 个样本第 h 个注意力头中，文本第 i 个 token 对图像第 j 个 patch 的相似度
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 若传入了文本 padding 掩码，则屏蔽 Query 中 padding 位置对应的注意力行
        # attention_mask 形状：(B, text_len)，1=有效 token，0=padding
        #
        # 广播推导（从右对齐）：
        #   scores           形状：(B, num_heads, text_len, img_patches)
        #   mask after expand形状：(B, 1,         text_len,  1          )
        #   广播结果         形状：(B, num_heads, text_len,  img_patches) ✓
        #
        # 注意：此处需扩展到 Query 维（dim -2），故用 unsqueeze(1).unsqueeze(-1) 而非 unsqueeze(1).unsqueeze(1)：
        #   unsqueeze(1).unsqueeze(1)  → (B, 1, 1, text_len)，最后一维=text_len 会与 img_patches=197 不等 → RuntimeError
        #   unsqueeze(1).unsqueeze(-1) → (B, 1, text_len, 1)，最后一维为 1 可广播到任意 img_patches ✓
        if attention_mask is not None:
            # unsqueeze 的作用是补齐 ndim，使 mask 与 scores 的维度数相同（均为 4 维），
            # 而非在此处发生广播：
            #   attention_mask：(B, text_len)            ndim=2
            #   .unsqueeze(1)  → (B, 1, text_len)       ndim=3
            #   .unsqueeze(-1) → (B, 1, text_len, 1)    ndim=4  ← 与 scores 的 ndim 对齐
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(-1)

            # masked_fill 内部才真正触发广播：
            #   scores         形状：(B, num_heads, text_len, img_patches)
            #   attention_mask 形状：(B, 1,         text_len, 1          )
            #                         ↑广播          ↑对齐      ↑广播
            # padding 位置（mask==0）对应行的所有 img_patches 列均置为 -∞，
            # softmax 后该 Query token 的注意力权重全部趋近 0
            scores = scores.masked_fill(attention_mask == 0, -1e9)

        # 沿 Key（img_patches）维度做 softmax，得到文本 token 对各图像 patch 的注意力概率分布
        # attention_weights 形状：(B, num_heads, text_len, img_patches)，每行之和为 1
        attention_weights = F.softmax(scores, dim=-1)

        # 以注意力权重对 V（图像 patch 特征）做加权求和，融合视觉信息到文本侧
        # attention 形状：(B, num_heads, text_len, head_dim)
        #   每个文本 token 已吸收来自图像的视觉信息
        attention = torch.matmul(attention_weights, V)
        return attention

### 1.5 多模态编码层（MultimodalEncoderLayer）

单层融合结构：**文本自注意力 → 图文交叉注意力 → FFN**，每子层均使用 Pre-LN + 残差连接。

In [13]:
class MultimodalEncoderLayer(nn.Module):
    """
    多模态编码器单层 = 文本自注意力 + 图文交叉注意力 + 前馈网络（FFN）。
    结构类似 BERT Encoder Layer，但增加了 Cross-Attention 注入图像信息。
    """

    def __init__(self, embed_dim=768, num_heads=12, ff_dim=3072, dropout=0.1):
        """
        初始化多模态编码器单层。
        Args:
            embed_dim (int): 输入/输出特征维度，默认 768
            num_heads (int): 多头注意力头数，默认 12
            ff_dim (int): 前馈网络中间层维度，BERT-base 标准为 3072（4×768），默认 3072
            dropout (float): 随机失活比例，防止过拟合，默认 0.1
        """
        super().__init__()  # 调用父类初始化，注册所有子模块
        # 文本内部自注意力模块（PyTorch 原生实现，内含 Q/K/V 投影 + 多头拼接 + 输出投影）
        # 参数说明：
        #   embed_dim   (int)  ：输入/输出特征维度，768；Q/K/V 投影矩阵形状均为 (embed_dim, embed_dim)
        #   num_heads   (int)  ：注意力头数，12；每头的维度 = embed_dim / num_heads = 64
        #   dropout     (float)：注意力权重上的 Dropout 概率，训练时随机置零部分权重以防过拟合
        #   batch_first (bool) ：为 True 时输入/输出形状为 (batch_size, seq_len, embed_dim)；
        #                        为 False 时（默认）形状为 (seq_len, batch_size, embed_dim)；
        #                        此处设为 True 以与 BERT 惯例保持一致
        # forward 调用方式：self_attention(query, key, value, key_padding_mask=...)
        #   返回 (attn_output, attn_weights)：
        #     attn_output  形状 (batch_size, seq_len, embed_dim)：各 token 融合上下文后的表示
        #     attn_weights 形状 (batch_size, seq_len, seq_len) ：各 token 对之间的注意力权重（可选）
        self.self_attention = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attention = MultiHeadCrossAttention(embed_dim, num_heads)  # 图文交叉注意力（文本 Query，图像 K/V）
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),   # 升维：768 → 3072
            nn.GELU(),                      # BERT 使用的激活函数（相比 ReLU 更平滑）
            nn.Dropout(dropout),            # FFN 内部 dropout
            nn.Linear(ff_dim, embed_dim),   # 降维回原维度：3072 → 768
            nn.Dropout(dropout),            # FFN 输出 dropout
        )
        self.norm1 = nn.LayerNorm(embed_dim)  # 自注意力前的层归一化（Pre-LN 风格）
        self.norm2 = nn.LayerNorm(embed_dim)  # 交叉注意力前的层归一化
        self.norm3 = nn.LayerNorm(embed_dim)  # FFN 前的层归一化
        self.dropout = nn.Dropout(dropout)    # 残差连接后的 dropout

    def forward(self, text_features, image_features, text_attention_mask=None):
        """
        前向传播：文本特征经过自注意力 + 交叉注意力 + FFN 融合图像信息。
        Args:
            text_features (Tensor): 文本 token 特征，形状 (batch_size, seq_len, embed_dim)
            image_features (Tensor): 图像 patch 特征，形状 (batch_size, num_patches, embed_dim)
            text_attention_mask (Tensor, optional): 文本 padding 掩码，形状 (batch_size, seq_len)
        Returns:
            text_features (Tensor): 融合后的多模态文本特征，形状 (batch_size, seq_len, embed_dim)
        """
        # ════════════════════════════════════════════════════════════
        # 子层 1：文本自注意力（Self-Attention）
        #   作用：让文本 token 相互感知上下文，捕捉句内语义依赖
        #   结构：Pre-LN → Self-Attn → Dropout → 残差相加
        # ════════════════════════════════════════════════════════════

        # 保存当前文本特征作为残差分支，供子层输出后做跳接
        # residual 形状：(batch_size, seq_len, embed_dim)
        #   - batch_size：批次内样本数量
        #   - seq_len   ：文本序列长度（含 [CLS]/[SEP]/padding）
        #   - embed_dim ：每个 token 的特征维度（768）
        residual = text_features

        # Pre-LayerNorm：对每个 token 的 embed_dim 维向量独立做均值/方差归一化
        # 将各维度拉到均值≈0、方差≈1，再通过可学习的 γ/β 仿射变换，稳定训练
        # 输入/输出形状不变：(batch_size, seq_len, embed_dim)
        text_features = self.norm1(text_features)

        # 将文本 padding 掩码转换为 PyTorch MultiheadAttention 所需的 key_padding_mask 格式
        # text_attention_mask 中：1=有效 token，0=padding；key_padding_mask 中：True=需屏蔽
        # 屏蔽后，padding 位置在注意力 softmax 前会被置为 -∞，确保不影响有效 token 的注意力分布
        if text_attention_mask is not None:
            key_padding_mask = (text_attention_mask == 0)  # 形状：(batch_size, seq_len)，bool 类型
        else:
            key_padding_mask = None  # 无 padding 时不屏蔽任何位置

        # Q=K=V=text_features：文本 token 对自身做注意力，捕捉 token 间语义依赖
        # text_attn_out 形状：(batch_size, seq_len, embed_dim)，每个 token 已吸收其余 token 的信息
        # _ 为注意力权重矩阵，形状 (batch_size, num_heads, seq_len, seq_len)，此处不使用故丢弃
        text_attn_out, _ = self.self_attention(
            text_features, text_features, text_features,
            key_padding_mask=key_padding_mask,
        )

        # 残差连接：将子层输出与输入跳接相加，缓解梯度消失，保留原始特征
        # self.dropout 以概率 p=dropout 随机将 text_attn_out 的元素置 0，防止过拟合
        # 输出形状：(batch_size, seq_len, embed_dim)
        text_features = residual + self.dropout(text_attn_out)

        # ════════════════════════════════════════════════════════════
        # 子层 2：图文交叉注意力（Cross-Attention）
        #   作用：文本 token 主动查询图像 patch 信息，实现视觉-语言融合（Fuse）
        #   结构：Pre-LN → Cross-Attn → Dropout → 残差相加
        # ════════════════════════════════════════════════════════════

        # 保存子层 1 输出作为本子层的残差分支
        # residual 形状：(batch_size, seq_len, embed_dim)
        residual = text_features

        # Pre-LayerNorm：进入 Cross-Attention 前归一化文本特征，稳定注意力计算
        # 输入/输出形状不变：(batch_size, seq_len, embed_dim)
        text_features = self.norm2(text_features)

        # 交叉注意力：文本 token 对图像 patch 做注意力查询
        #   query = text_features：文本 token，形状 (batch_size, seq_len, embed_dim)，充当 Q
        #   key   = image_features：图像 patch，形状 (batch_size, 197, embed_dim)，充当 K
        #   value = image_features：图像 patch，形状 (batch_size, 197, embed_dim)，充当 V
        # 每个文本 token 在 197 个图像 token 上计算注意力权重，加权求和得到视觉信息
        # cross_attn_out 形状：(batch_size, seq_len, embed_dim)，每个文本 token 已注入图像信息
        cross_attn_out = self.cross_attention(
            query=text_features,
            key=image_features,
            value=image_features,
        )

        # 残差连接：将图文融合结果与子层 2 输入跳接，保留原始文本特征并叠加视觉信息
        # 输出形状：(batch_size, seq_len, embed_dim)
        text_features = residual + self.dropout(cross_attn_out)

        # ════════════════════════════════════════════════════════════
        # 子层 3：前馈网络（FFN）
        #   作用：对每个 token 独立做非线性变换，提升表达能力
        #   结构：Pre-LN → Linear(768→3072) → GELU → Dropout → Linear(3072→768) → Dropout → 残差相加
        # ════════════════════════════════════════════════════════════

        # 保存子层 2 输出作为本子层的残差分支
        # residual 形状：(batch_size, seq_len, embed_dim)
        residual = text_features

        # Pre-LayerNorm：进入 FFN 前归一化，稳定升维阶段的数值范围
        # 输入/输出形状不变：(batch_size, seq_len, embed_dim)
        text_features = self.norm3(text_features)

        # FFN 前向：对每个 token 独立做 768→3072→768 的升维-降维非线性变换
        #   Linear(768→3072)：扩展特征表示空间（BERT 标准 4×倍扩展）
        #   GELU：平滑激活函数，比 ReLU 更适合 Transformer（在负区间有梯度）
        #   Dropout：FFN 内部正则化
        #   Linear(3072→768)：压缩回原始维度
        #   Dropout：FFN 输出正则化（已含在 nn.Sequential 中，无需额外 dropout）
        # ff_out 形状：(batch_size, seq_len, embed_dim=768)
        ff_out = self.feed_forward(text_features)

        # 残差连接：FFN 输出与输入跳接，此处不再额外加 dropout
        # （self.feed_forward 的 nn.Sequential 末尾已含 Dropout，不重复添加）
        # 输出形状：(batch_size, seq_len, embed_dim)，即本层最终的多模态融合文本特征
        text_features = residual + ff_out

        # 返回本层融合后的文本特征，作为下一层 MultimodalEncoderLayer 的输入（或最终输出）
        # 形状：(batch_size, seq_len, embed_dim)
        #   - batch_size：批次内样本数量
        #   - seq_len   ：文本序列长度（与输入相同，不改变 token 数量）
        #   - embed_dim ：每个 token 已融合本层图像信息的特征向量维度（768）
        return text_features

### 1.6 多模态编码器（MultimodalEncoder）

堆叠多层 `MultimodalEncoderLayer`，对应 ALBEF 论文 Fuse 阶段的 6 层融合编码器。

In [14]:
class MultimodalEncoder(nn.Module):
    """
    多模态编码器：堆叠多层 MultimodalEncoderLayer，逐层将图像信息融合进文本特征。
    对应 ALBEF 论文中 BERT 后 6 层（此处演示用 6 层）。
    """

    def __init__(self, num_layers=6, embed_dim=768, num_heads=12, ff_dim=3072, dropout=0.1):
        """
        初始化多模态编码器。
        Args:
            num_layers (int): 堆叠的 MultimodalEncoderLayer 层数，论文中为 6
            embed_dim (int): 输入/输出特征维度，需与 BERT 一致，默认 768
            num_heads (int): 多头注意力头数，默认 12
            ff_dim (int): 前馈网络中间层维度，默认 3072（4×embed_dim）
            dropout (float): 随机失活比例，防止过拟合
        """
        super().__init__()  # 调用父类初始化，注册层列表
        self.layers = nn.ModuleList([
            MultimodalEncoderLayer(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)  # 创建 num_layers 个融合层，存入 ModuleList 以便自动管理参数
        ])

    def forward(self, text_features, image_features, text_attention_mask=None):
        """
        逐层将图像信息融合进文本特征。
        Args:
            text_features (Tensor): 文本 token 特征，形状 (batch_size, seq_len, embed_dim)
            image_features (Tensor): 图像 patch 特征，形状 (batch_size, num_patches, embed_dim)
            text_attention_mask (Tensor, optional): 文本 padding 掩码，形状 (batch_size, seq_len)；
                                                    1 表示有效 token，0 表示 padding
        Returns:
            text_features (Tensor): 融合后的多模态文本特征，形状 (batch_size, seq_len, embed_dim)；
                                    每个 token 已通过多层 Cross-Attention 看到图像信息
        """
        # 依次通过每一层 MultimodalEncoderLayer，共 num_layers（默认 6）层
        # 每层内部结构：Self-Attention → Cross-Attention → FFN（均含 Pre-LN + 残差）
        # 注意：
        #   - text_features 在每层结束后被更新，融合了越来越多的图像语义
        #   - image_features 在所有层中保持不变（仅作为 Key/Value 被文本查询，自身不更新）
        #   这种设计使文本逐层"阅读"图像，类似 decoder cross-attention over encoder output
        for layer in self.layers:
            # layer 输入：text_features (batch_size, seq_len, embed_dim)
            #              image_features (batch_size, 197, embed_dim)  ← 固定不变
            # layer 输出：text_features (batch_size, seq_len, embed_dim)  ← 每层更新
            text_features = layer(text_features, image_features, text_attention_mask)

        # 返回经过全部 num_layers 层融合后的多模态文本特征
        # 形状：(batch_size, seq_len, embed_dim)
        #   - batch_size：批次内样本数量
        #   - seq_len   ：文本序列长度（不变，token 数量未改变）
        #   - embed_dim ：768，每个 token 的向量已通过 6 轮 Cross-Attention 深度融合图像信息
        # 第 0 位 [CLS] token 的向量可作为句级多模态表示，用于 ITM 分类头
        return text_features

### 1.7 ALBEF 完整模型（ALBEF）

整合图像编码器、文本编码器、多模态编码器及三个任务头，提供统一的前向接口。三个训练任务如下：

| 缩写 | 全称 | 中文 | 阶段 | 作用 |
|------|------|------|------|------|
| **ITC** | Image-Text Contrastive | 图文对比学习 | Align | 单模态独立编码后，拉近匹配图文的 [CLS] 向量、推远不匹配的，实现特征空间对齐 |
| **ITM** | Image-Text Matching | 图文匹配 | Fuse | 多模态融合后，用 [CLS] 向量做二分类，判断图文对是否真实匹配 |
| **MLM** | Masked Language Modeling | 掩码语言模型 | Fuse | 随机遮掩文本中 15% 的词，结合图像信息预测被遮掩的原始词 |

In [15]:
class ALBEF(nn.Module):
    """
    ALBEF 完整模型（教学简化版）。
    三大训练任务：
      - ITC（Image-Text Contrastive）：单模态编码后对比学习，Align 阶段
      - ITM（Image-Text Matching）：多模态融合后判断图文是否匹配
      - MLM（Masked Language Modeling）：多模态融合后预测被 mask 的词
    """

    def __init__(self, embed_dim=768, num_multimodal_layers=6):
        """
        初始化 ALBEF 完整模型，构建所有子模块。
        Args:
            embed_dim (int): 统一的特征维度，所有子模块共享，默认 768
            num_multimodal_layers (int): 多模态融合编码器的堆叠层数，论文为 6
        """
        super().__init__()  # 调用父类初始化，注册所有子模块
        self.image_encoder = ImageEncoder(embed_dim)       # 图像单模态编码器（ViT-B/16 骨干，输出 197 个 token）
        self.text_encoder = TextEncoder(embed_dim)         # 文本单模态编码器（BERT-base）
        self.multimodal_encoder = MultimodalEncoder(       # 多模态融合编码器（Fuse 阶段）
            num_layers=num_multimodal_layers,              # 融合层数，论文取 6
            embed_dim=embed_dim,                           # 特征维度，与图像/文本编码器对齐
        )  # 构建 6 层 Cross-Attention 融合编码器
        self.image_proj = nn.Linear(embed_dim, embed_dim)  # ITC：图像对比投影头（映射到对比空间）
        self.text_proj = nn.Linear(embed_dim, embed_dim)   # ITC：文本对比投影头（映射到对比空间）
        self.temperature = nn.Parameter(torch.ones([]) * 0.07)  # 可学习温度参数，缩放相似度分数
        self.itm_head = nn.Linear(embed_dim, 2)            # ITM：二分类头（0=不匹配, 1=匹配）
        self.mlm_head = nn.Linear(embed_dim, 30522)        # MLM：词表预测头，30522 为 BERT vocab 大小

    def forward(self, images, input_ids, attention_mask, task="contrastive", **kwargs):
        """
        统一前向入口，根据 task 参数分发到不同任务。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            task (str): 任务类型，"contrastive" | "itm" | "mlm"
            **kwargs: 透传给子任务的额外参数（如 labels、masked_input_ids 等）
        Returns:
            根据 task 不同，返回对应任务的损失（Tensor 标量）或 logits
        """
        if task == "contrastive":                          # ITC 对比学习任务分支
            return self.contrastive_forward(images, input_ids, attention_mask)  # 返回标量 InfoNCE 损失
        elif task == "itm":                               # ITM 图文匹配任务分支
            return self.itm_forward(images, input_ids, attention_mask, **kwargs)  # 返回二分类损失或 logits
        elif task == "mlm":                               # MLM 掩码语言模型任务分支
            return self.mlm_forward(images, input_ids, attention_mask, **kwargs)  # 返回词表预测损失或 logits
        else:                                             # 传入了不在白名单内的 task 字符串
            raise ValueError(f"不支持的任务类型: {task}")  # 传入非法 task 时抛出异常

    def encode_unimodal(self, images, input_ids, attention_mask):
        """
        单模态编码：图像和文本各自独立编码，不做 Cross-Attention。
        用于 ITC 对比学习（Align Before Fuse 中的 Align 阶段）。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
        Returns:
            image_features (Tensor): 图像 token 特征，形状 (batch_size, 197, embed_dim)；197 = [CLS]+196 patch
            text_features (Tensor): 文本 token 特征，形状 (batch_size, seq_len, embed_dim)
        """
        # 图像编码：PIL 图像列表 → patch token 特征序列
        # 内部流程：ViTImageProcessor 预处理 → ViT-B/16 Transformer → 线性投影
        # 输出形状：(batch_size, 197, embed_dim=768)
        #   - 197 = 1 个 [CLS] token（第 0 位，全局图像语义）+ 196 个 patch token（局部视觉特征）
        image_features = self.image_encoder(images)

        # 文本编码：token ID 序列 → 上下文向量序列
        # 内部流程：BERT-base 12 层 Transformer 编码
        # 输出形状：(batch_size, seq_len, embed_dim=768)
        #   - seq_len：文本序列长度（含 [CLS]/[SEP]/padding，≤ max_length）
        #   - 第 0 位 [CLS] token 的向量可作为句子级全局文本表示
        text_features = self.text_encoder(input_ids, attention_mask)

        # 返回两路单模态特征，此时图文尚未交互（Align Before Fuse 的 Align 阶段）
        # image_features：(batch_size, 197, 768)；text_features：(batch_size, seq_len, 768)
        return image_features, text_features

    def encode_multimodal(self, images, input_ids, attention_mask):
        """
        多模态编码：先单模态编码，再经多层 Cross-Attention 融合。
        用于 ITM 和 MLM（Fuse 阶段）。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
        Returns:
            multimodal_text_features (Tensor): 多模态融合后文本特征，形状 (batch_size, seq_len, embed_dim)
            image_features (Tensor): 图像 token 特征，形状 (batch_size, 197, embed_dim)；197 = [CLS]+196 patch
        """
        # 第一步：单模态独立编码（Align 阶段）
        # image_features：(batch_size, 197, embed_dim)；text_features：(batch_size, seq_len, embed_dim)
        image_features, text_features = self.encode_unimodal(images, input_ids, attention_mask)

        # 第二步：多模态 Cross-Attention 融合（Fuse 阶段）
        # 文本 token 作为 Query，图像 197 个 token 作为 Key/Value，经 6 层融合后文本吸收视觉信息
        # attention_mask 传入以屏蔽文本 padding 位置，图像侧无需掩码（无 padding）
        # multimodal_text_features 形状：(batch_size, seq_len, embed_dim)
        multimodal_text_features = self.multimodal_encoder(
            text_features, image_features, attention_mask
        )

        # 同时返回融合后文本特征和原始图像特征
        # multimodal_text_features：(batch_size, seq_len, embed_dim)，已融合视觉信息
        # image_features           ：(batch_size, 197, embed_dim)，保持不变，供 ITM/MLM 上层按需使用
        return multimodal_text_features, image_features

    def contrastive_forward(self, images, input_ids, attention_mask):
        """
        ITC 图文对比学习（InfoNCE 损失）。
        流程：单模态编码 → 池化全局向量 → 投影 → L2 归一化 → 计算相似度矩阵 → 对称交叉熵
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码
        Returns:
            loss (Tensor): 标量，图文双向 InfoNCE 对比损失的平均值
        """
        # 单模态独立编码，不做 Cross-Attention
        # image_features：(batch_size, 197, embed_dim)；text_features：(batch_size, seq_len, embed_dim)
        image_features, text_features = self.encode_unimodal(images, input_ids, attention_mask)

        # 取各模态 [CLS] token 作为全局语义向量（第 0 位）
        # image_embeds 形状：(batch_size, embed_dim=768)，代表全局图像语义
        # text_embeds  形状：(batch_size, embed_dim=768)，代表全局文本语义
        image_embeds = image_features[:, 0, :]
        text_embeds = text_features[:, 0, :]

        # 通过各自的线性投影头映射到对比学习空间
        # 投影后形状不变：(batch_size, embed_dim)，但特征已被变换到更适合度量的子空间
        image_embeds = self.image_proj(image_embeds)
        text_embeds = self.text_proj(text_embeds)

        # L2 归一化，将向量约束到单位球面
        # 归一化后点积等价于余弦相似度，取值范围 [-1, 1]
        # 形状不变：(batch_size, embed_dim)
        image_embeds = F.normalize(image_embeds, dim=-1)
        text_embeds = F.normalize(text_embeds, dim=-1)

        # 计算图文相似度矩阵并缩放
        # image_embeds @ text_embeds.T：每行 i 为第 i 张图与全部 batch_size 段文本的点积相似度
        # / self.temperature（可学习，初始 0.07）：放大分数差异，使 softmax 分布更尖锐
        # logits 形状：(batch_size, batch_size)，logits[i,j] = 图 i 与文本 j 的相似度分数
        logits = torch.matmul(image_embeds, text_embeds.t()) / self.temperature

        # 构造正样本标签：第 i 张图的正样本为第 i 段文本（batch 内对角线对齐）
        # labels 形状：(batch_size,)，值为 [0, 1, 2, ..., batch_size-1]
        labels = torch.arange(logits.size(0), device=logits.device)

        # 图像→文本方向：以每行（图像）为 query，文本为候选，计算交叉熵
        # loss_i 形状：标量；期望 logits[i, i] 最大，即图 i 最相似于文本 i
        loss_i = F.cross_entropy(logits, labels)

        # 文本→图像方向：对称地，以每列（文本）为 query，图像为候选
        # logits.t() 形状：(batch_size, batch_size)，行列互换后文本变为 query
        loss_t = F.cross_entropy(logits.t(), labels)

        # 双向对称损失取平均，得到最终 InfoNCE 对比损失（标量）
        loss = (loss_i + loss_t) / 2
        return loss

    def itm_forward(self, images, input_ids, attention_mask, labels=None):
        """
        ITM 图文匹配：判断图文是否匹配（二分类）。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码
            labels (Tensor, optional): 形状 (batch_size,)，1=匹配, 0=不匹配；为 None 时只返回 logits
        Returns:
            loss (Tensor): 有 labels 时返回标量二分类交叉熵损失
            itm_logits (Tensor): 无 labels 时返回形状 (batch_size, 2) 的分类 logits
        """
        # 先单模态编码，再经 6 层 Cross-Attention 融合图像信息
        # multimodal_features 形状：(batch_size, seq_len, embed_dim=768)，文本侧融合了视觉信息
        # _ 为图像特征 (batch_size, 197, embed_dim)，ITM 不需要，用 _ 丢弃
        multimodal_features, _ = self.encode_multimodal(images, input_ids, attention_mask)

        # 取融合后序列第 0 位（[CLS] token）作为整段多模态句子的全局表示
        # cls_features 形状：(batch_size, embed_dim=768)
        #   [CLS] token 经过 Self-Attn 和 Cross-Attn 后已聚合了全文和全图的语义
        cls_features = multimodal_features[:, 0, :]

        # 通过二分类线性头预测图文是否匹配
        # itm_logits 形状：(batch_size, 2)
        #   - 第 0 维：不匹配（negative）的 logit
        #   - 第 1 维：匹配（positive）的 logit
        itm_logits = self.itm_head(cls_features)

        if labels is not None:  # 训练模式：提供真实标签，计算损失
            # labels 形状：(batch_size,)，值为 0（不匹配）或 1（匹配）
            # 返回标量二分类交叉熵损失
            loss = F.cross_entropy(itm_logits, labels)
            return loss
        else:  # 推理模式：未提供标签，直接返回 logits 供调用方 softmax/argmax
            # itm_logits 形状：(batch_size, 2)
            return itm_logits

    def mlm_forward(self, images, input_ids, attention_mask, masked_input_ids=None, mlm_labels=None):
        """
        MLM 掩码语言模型：随机 mask 部分词，结合图像信息预测原词。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 原始文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码
            masked_input_ids (Tensor, optional): 含 [MASK] 的 token ID；为 None 时自动生成
            mlm_labels (Tensor, optional): 被 mask 位置的原始 token ID，其余为 -100
        Returns:
            loss (Tensor): 有 labels 时返回标量词表预测交叉熵损失
            mlm_logits (Tensor): 无 labels 时返回形状 (B, seq_len, vocab_size) 的 logits
        """
        # 若未提供预处理好的 mask 样本，则自动生成
        # mask_tokens 按 BERT 规则随机选 15% token，80% 换 [MASK]、10% 换随机词、10% 保持不变
        if masked_input_ids is None and mlm_labels is None:
            # masked_input_ids 形状：(batch_size, seq_len)，含 [MASK]（token ID=103）的 token 序列
            # mlm_labels       形状：(batch_size, seq_len)，被 mask 位置为原始 token ID，其余为 -100
            masked_input_ids, mlm_labels = self.mask_tokens(input_ids)

        # 用含 [MASK] 的序列做多模态融合编码
        # 注意：传入的是 masked_input_ids 而非原始 input_ids，BERT 需要看到 [MASK] 才能预测被遮掩的词
        # multimodal_features 形状：(batch_size, seq_len, embed_dim=768)，已融合图像信息
        # _ 为图像特征，MLM 不需要，丢弃
        multimodal_features, _ = self.encode_multimodal(images, masked_input_ids, attention_mask)

        # 对每个 token 位置独立预测词表中的原始词
        # mlm_logits 形状：(batch_size, seq_len, vocab_size=30522)
        #   - seq_len   ：每个位置都输出词表 logits（非 mask 位置的预测在损失中被 ignore_index 忽略）
        #   - vocab_size：BERT 词表大小（bert-base-uncased 为 30522）
        mlm_logits = self.mlm_head(multimodal_features)

        if mlm_labels is not None:  # 训练模式：提供真实标签，计算损失
            loss = F.cross_entropy(
                mlm_logits.view(-1, mlm_logits.size(-1)),  # 展平为 (batch_size×seq_len, vocab_size)
                mlm_labels.view(-1),                       # 展平为 (batch_size×seq_len,)
                ignore_index=-100,                         # 非 mask 位置标签为 -100，F.cross_entropy 自动跳过
            )
            # 返回标量词表预测交叉熵损失（仅被 mask 位置贡献损失）
            return loss
        else:  # 推理模式：未提供标签，返回 logits 供调用方 argmax/topk
            # mlm_logits 形状：(batch_size, seq_len, vocab_size=30522)
            return mlm_logits

    def mask_tokens(self, input_ids, mlm_probability=0.15):
        """
        按 BERT 规则创建 MLM 训练样本。
        15% token 被选中；其中 80% 换 [MASK]，10% 换随机词，10% 保持不变。
        Args:
            input_ids (Tensor): 原始 token ID，形状 (batch_size, seq_len)
            mlm_probability (float): 被 mask 的 token 比例，BERT 标准为 0.15
        Returns:
            input_ids (Tensor): 含 [MASK] 的 token ID，形状 (batch_size, seq_len)
            labels (Tensor): 被 mask 位置的原始 ID，其余为 -100，形状 (batch_size, seq_len)
        """
        device = input_ids.device                  # 与输入在同一设备
        labels = input_ids.clone()                 # 复制一份作为预测标签，后续在非 mask 位置写 -100
        probability_matrix = torch.full(labels.shape, mlm_probability, device=device)  # 全表初始化为 mask 概率
        special_tokens_mask = torch.zeros_like(input_ids, dtype=torch.bool, device=device)  # 特殊 token 位置标记矩阵，初始全 False
        for special_id in [0, 101, 102]:           # PAD=0, [CLS]=101, [SEP]=102，不参与 mask
            special_tokens_mask = special_tokens_mask | (input_ids == special_id)  # 逐一标记特殊 token 位置为 True
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)  # 特殊 token 的 mask 概率置 0
        masked_indices = torch.bernoulli(probability_matrix).bool()       # 按概率采样待 mask 位置
        labels[~masked_indices] = -100             # 非 mask 位置标签设为 -100（损失中忽略）
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8, device=device)).bool() & masked_indices  # 80% 的被选中 token：替换为 [MASK]
        input_ids[indices_replaced] = 103          # 80% 的 mask 位置替换为 [MASK]（token ID=103）
        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5, device=device)).bool() & masked_indices & ~indices_replaced  # 剩余 20% 中再取 50%：替换为随机词
        random_words = torch.randint(0, 30522, labels.shape, device=device)  # 随机词 ID（vocab_size=30522）
        input_ids[indices_random] = random_words[indices_random]             # 10% 的位置替换为随机词
        return input_ids, labels  # 返回含 [MASK] 的 token ID 序列 和 对应标签（非 mask 位置为 -100）

### 1.8 三任务演示（ITC / ITM / MLM）

实例化完整 ALBEF 模型，依次调用三种训练任务的前向传播，并打印特征形状与损失值。

In [16]:
def demo_albef_with_cross_attention():
    """
    演示函数：依次展示单模态编码、多模态融合、ITC/ITM/MLM 三个任务。
    使用随机图像 + 4 条英文句子，无需下载数据集。
    """
    print("=" * 80)                              # 打印顶部分隔线，美化输出
    print("ALBEF模型 - 包含交叉注意力机制")        # 演示标题
    print("=" * 80)                              # 打印分隔线

    model = ALBEF(embed_dim=768, num_multimodal_layers=6)  # 实例化完整 ALBEF 模型
    model.eval()  # 评估模式：关闭 dropout，固定 BN 统计量

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # 加载 BERT 分词器
    texts = [                                    # 4 条英文演示句子，与 4 张随机图像一一对应
        "A cat sitting on a chair",
        "A dog running in the park",
        "A bird flying in the sky",
        "A fish swimming in water",
    ]
    # 批量分词并填充到相同长度
    # padding=True：将 batch 内所有句子 padding 到最长句的长度（用 [PAD]=0 填充）
    # truncation=True：超过 max_length=32 时截断
    # return_tensors='pt'：返回 torch.Tensor
    # 返回 dict，主要字段：
    #   - input_ids      形状 (4, seq_len)：每个 token 的词表 ID（seq_len ≤ 32）
    #   - attention_mask 形状 (4, seq_len)：1=有效 token，0=padding 位置
    tokenized = tokenizer(
        texts, padding=True, truncation=True, return_tensors='pt', max_length=32
    )
    # 生成 4 张随机 RGB PIL 图像（演示用，无需真实图片）
    # np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)：形状 (H=224, W=224, C=3)，像素值范围 [0,255]
    # Image.fromarray(...)：将 numpy uint8 数组转为 PIL.Image.Image，供 ImageEncoder.forward() 内的 ViTImageProcessor 处理
    images = [Image.fromarray(np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)) for _ in range(4)]

    print("输入数据形状:")                        # 打印输入数据维度信息
    print(f"  图像: {len(images)} 张 PIL.Image（224×224 RGB，演示用随机像素）")  # 4 张 PIL 图像，processor 内部 resize+归一化
    print(f"  文本ID: {tokenized['input_ids'].shape}")         # (4, seq_len)
    print(f"  注意力掩码: {tokenized['attention_mask'].shape}")  # (4, seq_len)
    print()                                      # 空行，分隔各输出块

    with torch.no_grad():  # 演示推理不需要梯度，节省显存
        print("1. 单模态编码")                   # 单模态编码阶段标题
        print("-" * 40)                          # 打印小节分隔线
        image_features, text_features = model.encode_unimodal(
            images, tokenized['input_ids'], tokenized['attention_mask']
        )  # 图像→(B,197,768)（[CLS]+196 patch token），文本→(B,seq,768)；各自独立编码，不做 Cross-Attention
        print(f"  图像特征形状: {image_features.shape}")   # (4, 197, 768)
        print(f"  文本特征形状: {text_features.shape}")    # (4, seq_len, 768)
        print()                                  # 空行，分隔各任务输出

        print("2. 多模态编码（交叉注意力）")       # 多模态融合阶段标题
        print("-" * 40)                          # 打印小节分隔线
        multimodal_features, _ = model.encode_multimodal(
            images, tokenized['input_ids'], tokenized['attention_mask']
        )  # 先单模态编码，再经多层 Cross-Attention 融合，返回 (B, seq_len, 768)
        print(f"  多模态特征形状: {multimodal_features.shape}")  # (4, seq_len, 768)
        print("  文本特征已通过交叉注意力与图像特征融合")  # 说明融合完成
        print()                                  # 空行

        print("3. 对比学习任务 (ITC)")            # ITC 任务标题
        print("-" * 40)                          # 打印小节分隔线
        contrastive_loss = model(
            images, tokenized['input_ids'], tokenized['attention_mask'], task="contrastive"
        )  # ITC 前向：单模态编码 → 投影 → L2 归一化 → 双向 InfoNCE 对比损失
        print(f"  对比学习损失: {contrastive_loss.item():.4f}")  # InfoNCE 标量损失
        print()                                  # 空行

        print("4. 图像-文本匹配任务 (ITM)")        # ITM 任务标题
        print("-" * 40)                          # 打印小节分隔线
        # 构造 ITM 标签：1=图文匹配，0=图文不匹配（演示用，实际由数据集决定）
        # 形状：(4,)；前 2 个样本标记为匹配，后 2 个标记为不匹配
        itm_labels = torch.tensor([1, 1, 0, 0])
        itm_loss = model(
            images, tokenized['input_ids'], tokenized['attention_mask'],
            task="itm", labels=itm_labels,
        )  # ITM 前向：多模态融合后取 [CLS] 向量，二分类交叉熵损失
        print(f"  ITM损失: {itm_loss.item():.4f}")  # 二分类交叉熵损失
        itm_logits = model.itm_forward(images, tokenized['input_ids'], tokenized['attention_mask'])  # 推理模式（无 labels）：返回 logits，形状 (4, 2)
        itm_predictions = torch.softmax(itm_logits, dim=-1)  # 转为概率，形状 (4, 2)
        print("  ITM预测概率:")                    # 打印每个样本的匹配概率
        for i, pred in enumerate(itm_predictions):  # 遍历 batch 中每个样本的预测结果
            print(f"    样本{i+1}: 不匹配={pred[0]:.3f}, 匹配={pred[1]:.3f}")  # pred[0]=不匹配, pred[1]=匹配
        print()                                  # 空行

        print("5. 掩码语言模型任务 (MLM)")         # MLM 任务标题
        print("-" * 40)                          # 打印小节分隔线
        mlm_loss = model(images, tokenized['input_ids'], tokenized['attention_mask'], task="mlm")  # MLM 前向：自动生成 mask，结合图像预测被 mask 词，返回标量损失
        print(f"  MLM损失: {mlm_loss.item():.4f}")  # 词表预测交叉熵损失
        print("  MLM预测基于图像-文本交叉注意力特征")  # 说明 MLM 使用多模态融合特征
        print()                                  # 空行

        print("6. 模型架构特点")                  # 模型统计信息标题
        print("-" * 40)                          # 打印小节分隔线
        total_params = sum(p.numel() for p in model.parameters())  # 统计全部可学习参数量
        multimodal_params = sum(p.numel() for p in model.multimodal_encoder.parameters())  # 仅多模态编码器参数量
        print(f"  总参数量: {total_params:,}")                              # 打印总参数量（逗号分隔便于阅读）
        print(f"  多模态编码器参数: {multimodal_params:,}")                 # 打印多模态编码器参数量
        print(f"  交叉注意力层数: {len(model.multimodal_encoder.layers)}")  # 打印 Cross-Attention 堆叠层数
        print("  包含图像-文本交叉注意力机制")      # 说明架构包含跨模态注意力
        print("  符合ALBEF论文架构")               # 与论文对齐说明

    print("=" * 80)                              # 打印底部分隔线
    print("演示完成！")                           # 演示结束提示
    print("=" * 80)                              # 打印分隔线


if __name__ == "__main__":                       # 直接运行脚本时执行（Jupyter 中不生效，保留工程规范）
    torch.manual_seed(42)    # 固定 PyTorch 随机种子，结果可复现
    np.random.seed(42)       # 固定 NumPy 随机种子
    demo_albef_with_cross_attention()  # 运行完整演示

ALBEF模型 - 包含交叉注意力机制


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


输入数据形状:
  图像: 4 张 PIL.Image（224×224 RGB，演示用随机像素）
  文本ID: torch.Size([4, 8])
  注意力掩码: torch.Size([4, 8])

1. 单模态编码
----------------------------------------
  图像特征形状: torch.Size([4, 197, 768])
  文本特征形状: torch.Size([4, 8, 768])

2. 多模态编码（交叉注意力）
----------------------------------------
  多模态特征形状: torch.Size([4, 8, 768])
  文本特征已通过交叉注意力与图像特征融合

3. 对比学习任务 (ITC)
----------------------------------------
  对比学习损失: 1.4030

4. 图像-文本匹配任务 (ITM)
----------------------------------------
  ITM损失: 0.9729
  ITM预测概率:
    样本1: 不匹配=0.771, 匹配=0.229
    样本2: 不匹配=0.685, 匹配=0.315
    样本3: 不匹配=0.636, 匹配=0.364
    样本4: 不匹配=0.445, 匹配=0.555

5. 掩码语言模型任务 (MLM)
----------------------------------------
  MLM损失: 11.0216
  MLM预测基于图像-文本交叉注意力特征

6. 模型架构特点
----------------------------------------
  总参数量: 277,826,877
  多模态编码器参数: 56,710,656
  交叉注意力层数: 6
  包含图像-文本交叉注意力机制
  符合ALBEF论文架构
演示完成！


## 二、动量编码器 + 负样本队列 + 动量蒸馏（ITC 增强）

本节在基础架构之上，为 ITC 对比学习增加以下三项机制（对应 ALBEF 论文 3.3-3.5 节）：

| 机制 | 作用 | 类比 |
|------|------|------|
| **动量编码器（Momentum Encoder）** | EMA 缓慢更新的 Teacher，提供稳定的特征 | MoCo 中的 key encoder |
| **负样本队列（Feature Queue）** | 存储历史动量特征，扩充 ITC 负样本池 | MoCo 中的 queue |
| **动量蒸馏（Momentum Distillation）** | KL 散度让 Student 学习 Teacher 的软概率分布 | 知识蒸馏 |

> 注意：动量机制**仅用于 ITC**，ITM 和 MLM 仍使用 Student 主模型前向传播。

### 2.1 特征队列（FeatureQueue）与动量更新函数（momentum_update）

`FeatureQueue`：环形 FIFO 缓冲区，存储历史 Teacher 特征作为额外负样本。  
`momentum_update`：EMA 更新函数，将 Student 参数缓慢同步到 Teacher。

In [ ]:
# ============================================================
# ALBEF 教学版 Notebook - 特征队列与动量更新
# 内容：FeatureQueue（负样本存储）+ momentum_update（EMA 更新）
# 依赖：需先运行依赖导入 + 各模型类 cell
# ============================================================

import copy  # 深拷贝动量编码器参数（Teacher 初始与 Student 相同）


class FeatureQueue:
    """
    FIFO 特征队列：存放动量编码器输出的历史特征，作为 ITC 额外负样本。
    思路来自 MoCo；论文队列大小 65536，教学演示默认 1024。
    """

    def __init__(self, queue_size=1024, embed_dim=768):
        """
        初始化特征队列（仅记录参数，队列 tensor 延迟到 _init_queue 中分配）。
        Args:
            queue_size (int): 队列最大容量（特征条数），论文为 65536，演示默认 1024
            embed_dim (int): 每条特征的维度（与 ITC 投影后维度一致），默认 768
        """
        self.queue_size = queue_size  # 队列容量
        self.embed_dim = embed_dim    # 特征维度

    def _init_queue(self, device):
        """
        首次使用时，在指定设备（CPU/GPU）上分配队列张量。
        Args:
            device (torch.device): 目标设备，与模型输入保持一致
        """
        self.queue = torch.zeros(self.queue_size, self.embed_dim, device=device)  # 预分配零值队列张量；形状 (queue_size, embed_dim)：queue_size=最多存储的历史特征条数，embed_dim=每条 L2 归一化特征的向量维度；全零初始化，等待后续 enqueue 写入
        self.ptr = 0          # 写入指针（int），指向下一个可写入槽位；初始为 0，随 enqueue 调用递增
        self.is_full = False  # 队列是否已被写满过（bool）；True 表示所有 queue_size 个槽位均有效，用于 get_queue 时决定返回全部还是部分

    def enqueue(self, features):
        """
        将当前 batch 的动量特征写入队列（环形缓冲区，detach 避免梯度回传）。
        Args:
            features (Tensor): 形状 (batch_size, embed_dim)，已 L2 归一化的动量特征
        """
        batch_size = features.shape[0]                            # 当前 batch 大小（int），即本次要写入的特征条数
        if self.ptr + batch_size <= self.queue_size:              # 判断：当前写入指针 + 本次 batch 大小 ≤ 队列总容量，剩余空间足够
            # 剩余空间足够：顺序写入
            self.queue[self.ptr:self.ptr + batch_size] = features.detach()  # 将本次 batch 特征（形状 (batch_size, embed_dim)）写入队列对应槽位；detach() 切断梯度
            self.ptr += batch_size                                # 指针右移 batch_size 步，指向下一个可写槽位
        else:                                                     # 剩余空间不足：需要环绕写入队列头部（循环覆盖）
            # 空间不足：尾部写满后从头覆盖（环形队列）
            remain = self.queue_size - self.ptr                   # 尾部剩余槽位数（int）：队列总容量 - 当前写入指针位置
            self.queue[self.ptr:] = features[:remain].detach()   # 写入尾部剩余槽位；features[:remain] 形状 (remain, embed_dim)
            self.queue[:batch_size - remain] = features[remain:].detach()  # 从队列头部继续写入剩余特征；features[remain:] 形状 (batch_size-remain, embed_dim)
            self.ptr = batch_size - remain                        # 更新写入指针到队列头部写完之后的位置（int）
            self.is_full = True                                   # 标记队列已被完整写满过，get_queue 将返回全部 queue_size 条特征

    def get_queue(self):
        """
        返回当前队列中所有有效特征，供 ITC 计算负样本相似度。
        Returns:
            Tensor: 有效特征张量，形状 (valid_size, embed_dim)；
                    valid_size：队列未满时为 ptr（当前写入位置），满后等于 queue_size
        """
        if self.is_full:                # 队列已至少被写满一次，所有位置均有效
            return self.queue           # 队列已满，返回全部 queue_size 条特征
        return self.queue[:self.ptr]    # 未满，只返回 ptr 之前已写入的部分


@torch.no_grad()  # EMA 更新不需要梯度
def momentum_update(student_module, momentum_module, momentum=0.995):
    """
    指数移动平均（EMA）更新动量编码器（Teacher）参数。
    公式：θ_m ← λ·θ_m + (1-λ)·θ_s
    Args:
        student_module (nn.Module): Student 子模块（参与梯度更新）
        momentum_module (nn.Module): Momentum 子模块（缓慢跟随 Student）
        momentum (float): λ，EMA 系数，论文取 0.995；越大 Teacher 越稳定
    """
    for param_s, param_m in zip(student_module.parameters(), momentum_module.parameters()):  # 同步遍历 Student 与 Teacher 的对应参数
        param_m.data = param_m.data * momentum + param_s.data * (1.0 - momentum)  # EMA 公式更新 Teacher 参数


### 2.2 带动量的 ALBEF（ALBEFWithMomentum）

在 Student-Teacher 框架下，整合负样本队列与动量蒸馏损失，实现 ALBEF 论文 3.3–3.5 节的完整 ITC 增强机制。

In [ ]:
class ALBEFWithMomentum(nn.Module):
    """
    在 ALBEF 基础上为 ITC 增加：
      1. 动量编码器（Teacher，EMA 更新，参数缓慢变化）
      2. 图文双队列（存历史 Teacher 特征，扩充 ITC 负样本）
      3. 动量蒸馏（KL 散度，让 Student 学习 Teacher 的软标签，缓解噪声图文对）
    注意：动量机制仅用于 ITC，ITM/MLM 仍走 Student 主模型。
    相对复杂，可以不掌握。
    """

    def __init__(self, base_model=None, queue_size=1024, momentum=0.995, embed_dim=768):
        """
        初始化带动量机制的 ALBEF 模型。
        Args:
            base_model (ALBEF, optional): 预构建的 Student 模型；为 None 时自动创建新 ALBEF 实例
            queue_size (int): 负样本特征队列容量，论文为 65536，教学演示默认 1024
            momentum (float): EMA 动量系数 λ，越大 Teacher 更新越慢、特征越稳定，论文取 0.995
            embed_dim (int): 特征维度，须与 Student 模型保持一致，默认 768
        """
        super().__init__()  # 调用父类 nn.Module 初始化，注册所有子模块
        self.student = base_model if base_model is not None else ALBEF(embed_dim=embed_dim)  # Student 主模型（有梯度，参与训练）
        self.momentum = momentum      # EMA 系数 λ，控制 Teacher 更新速度
        self.queue_size = queue_size  # 负样本队列容量

        # 深拷贝 ITC 相关模块作为动量编码器（Teacher 初始与 Student 完全相同）
        self.momentum_image_encoder = copy.deepcopy(self.student.image_encoder)  # Teacher 图像编码器
        self.momentum_text_encoder = copy.deepcopy(self.student.text_encoder)    # Teacher 文本编码器
        self.momentum_image_proj = copy.deepcopy(self.student.image_proj)        # Teacher 图像投影头
        self.momentum_text_proj = copy.deepcopy(self.student.text_proj)          # Teacher 文本投影头

        # Teacher 不参与反向传播，冻结参数以节省显存
        for module in [                              # 待冻结的 Teacher 子模块列表（共 4 个，与 Student 侧的 4 个 ITC 模块一一对应）
            self.momentum_image_encoder,             # Teacher 图像编码器（ViT-B/16，对应 Student 的 image_encoder）
            self.momentum_text_encoder,              # Teacher 文本编码器（BERT-base，对应 Student 的 text_encoder）
            self.momentum_image_proj,                # Teacher 图像 ITC 投影头（Linear，对应 Student 的 image_proj）
            self.momentum_text_proj,                 # Teacher 文本 ITC 投影头（Linear，对应 Student 的 text_proj）
        ]:
            for param in module.parameters():        # 遍历该 Teacher 子模块的所有参数张量（权重矩阵、偏置等）
                param.requires_grad = False          # 将 requires_grad 置 False：冻结 Teacher 参数，使其不接受梯度，只通过 EMA 缓慢更新

        self.image_queue = FeatureQueue(queue_size=queue_size, embed_dim=embed_dim)  # 图像特征 FIFO 队列（负样本池）
        self.text_queue = FeatureQueue(queue_size=queue_size, embed_dim=embed_dim)   # 文本特征 FIFO 队列（负样本池）

    def _init_queues(self, device):
        """
        将队列张量初始化到与输入相同的设备上。
        Args:
            device (torch.device): 目标设备（CPU 或 GPU），与模型输入保持一致
        """
        self.image_queue._init_queue(device)  # 初始化图像队列 tensor 到指定设备
        self.text_queue._init_queue(device)   # 初始化文本队列 tensor 到指定设备

    def encode_itc(self, images, input_ids, attention_mask, use_momentum=False):
        """
        提取 ITC 用的 L2 归一化全局向量（可选 Student 或 Teacher）。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            use_momentum (bool): True 使用 Teacher（动量编码器），False 使用 Student
        Returns:
            image_embeds (Tensor): 图像全局向量（L2 归一化），形状 (batch_size, embed_dim)
            text_embeds (Tensor): 文本全局向量（L2 归一化），形状 (batch_size, embed_dim)
        """
        if use_momentum:                                     # 如果使用动量编码器（Teacher）
            image_encoder = self.momentum_image_encoder     # 用动量版的图像编码器
            text_encoder = self.momentum_text_encoder       # 用动量版的文本编码器
            image_proj = self.momentum_image_proj           # 用动量版的图像投影头
            text_proj = self.momentum_text_proj             # 用动量版的文本投影头
        else:                                              # 使用 Student 主模型
            image_encoder = self.student.image_encoder      # 用 Student 的图像编码器
            text_encoder = self.student.text_encoder        # 用 Student 的文本编码器
            image_proj = self.student.image_proj            # 用 Student 的图像投影头
            text_proj = self.student.text_proj              # 用 Student 的文本投影头

        image_features = image_encoder(images)                                         # 图像编码：形状 (batch_size, 197, embed_dim)；197 = 1个[CLS]全局token + 196个patch token（14×14切片）；第 0 位 [CLS] 代表全图语义，embed_dim=768
        text_features = text_encoder(input_ids, attention_mask)                       # 文本编码：形状 (batch_size, seq_len, embed_dim)；seq_len=含 padding 的 token 序列长度；embed_dim=768
        image_embeds = F.normalize(image_proj(image_features[:, 0, :]), dim=-1)      # 取 ViT [CLS] token（第 0 位，形状 (batch_size, embed_dim)）→ 线性投影头 → L2 归一化（dim=-1 对最后一维归一化）；输出形状 (batch_size, embed_dim)
        text_embeds = F.normalize(text_proj(text_features[:, 0, :]), dim=-1)         # 取 BERT [CLS] token（第 0 位，形状 (batch_size, embed_dim)）→ 线性投影头 → L2 归一化；输出形状 (batch_size, embed_dim)；与图像向量在同一语义空间做点积对比
        return image_embeds, text_embeds  # 返回图文全局向量二元组；image_embeds 形状 (batch_size, embed_dim)，text_embeds 形状 (batch_size, embed_dim)，均已 L2 归一化，直接点积即余弦相似度

    def contrastive_forward_with_momentum(self, images, input_ids, attention_mask, alpha=0.4):
        """
        带队列与动量蒸馏的 ITC 损失。
        total_loss = (1-α)·InfoNCE + α·KL(Student || Teacher)
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            alpha (float): 蒸馏损失权重，论文取 0.4
        Returns:
            dict: 损失字典，键包含 total_loss / contrastive_loss / distill_loss / queue_text_size / queue_image_size
        """
        device = images.device                                                # 当前设备（CPU/GPU）
        if not hasattr(self.image_queue, 'queue'):                            # 若队列未初始化（首次调用）
            self._init_queues(device)                                         # 将队列 tensor 放到正确设备

        batch_size = images.size(0)                                           # 当前 batch 大小
        temperature = self.student.temperature                                # InfoNCE 温度超参（可学习）

        # ---- Student 特征（有梯度，参与优化）----
        image_embeds_s, text_embeds_s = self.encode_itc(
            images, input_ids, attention_mask, use_momentum=False             # 提取 Student 图文全局特征；返回 image_embeds_s (batch_size, embed_dim) 和 text_embeds_s (batch_size, embed_dim)，均已 L2 归一化，携带梯度用于反向传播
        )

        # ---- Momentum（Teacher）特征（无梯度）+ 写入队列 ----
        with torch.no_grad():                                                 # Teacher 前向与入队均不开梯度，Teacher 参数仅通过 EMA 更新
            image_embeds_m, text_embeds_m = self.encode_itc(
                images, input_ids, attention_mask, use_momentum=True          # 提取 Teacher 图文全局特征；返回 image_embeds_m (batch_size, embed_dim) 和 text_embeds_m (batch_size, embed_dim)；用作正样本配对对象及蒸馏软标签来源
            )
            self.image_queue.enqueue(image_embeds_m)                          # 图像 Teacher 特征入队，扩充负样本
            self.text_queue.enqueue(text_embeds_m)                            # 文本 Teacher 特征入队，扩充负样本

        queue_images = self.image_queue.get_queue()                           # 历史图像 Teacher 特征；形状 (Q, embed_dim)：Q=队列当前有效特征条数（队列满时=queue_size，未满时=ptr），embed_dim=768；供 t2i 方向计算纯负样本相似度
        queue_texts = self.text_queue.get_queue()                             # 历史文本 Teacher 特征；形状 (Q, embed_dim)：Q=队列当前有效特征条数，embed_dim=768；供 i2t 方向计算纯负样本相似度

        # ---- 图像→文本 InfoNCE：正样本=batch 内配对，负样本=队列 ----
        pos_i2t = torch.matmul(image_embeds_s, text_embeds_m.T) / temperature    # Student 图像 VS batch 内 Teacher 文本（含正例）；返回 (batch_size, batch_size)，即 B×B 的图像-文本相似度矩阵，对角线位置为正样本
        neg_i2t = torch.matmul(image_embeds_s, queue_texts.T) / temperature if queue_texts.numel() > 0 else None  # Student 图像 VS 队列历史文本（纯负例）；返回 (batch_size, Q)，Q=队列当前有效特征数；队列空时为 None
        if neg_i2t is not None and neg_i2t.shape[1] > 0:  # 队列不为空时，将 batch 内正负样本矩阵与队列负样本矩阵在列方向（dim=1）拼接
            logits_i2t = torch.cat([pos_i2t, neg_i2t], dim=1)                    # 合并后形状 (batch_size, batch_size+Q)：前 batch_size 列为 batch 内候选（含正例），后 Q 列为历史队列纯负样本
        else:                                                                      # 队列尚空（训练初始几步尚未积累历史特征）
            logits_i2t = pos_i2t                                                 # 仅使用 batch 内相似度矩阵；形状 (batch_size, batch_size)
        labels_i2t = torch.arange(batch_size, device=device)                     # 正样本索引标签；形状 (batch_size,)；labels_i2t[i]=i 表示第 i 张图的正例是 logits_i2t 第 i 列（对角线）
        loss_i2t = F.cross_entropy(logits_i2t, labels_i2t)                      # 图像→文本 InfoNCE 对比损失（标量）；cross_entropy 对每行 logits 做 softmax 后计算负对数似然

        # ---- 文本→图像 InfoNCE（对称方向）----
        pos_t2i = torch.matmul(text_embeds_s, image_embeds_m.T) / temperature    # Student 文本 VS batch 内 Teacher 图像（含正例）；返回 (batch_size, batch_size)，即 B×B 的文本-图像相似度矩阵（i2t 方向的对称操作），对角线为正样本
        neg_t2i = torch.matmul(text_embeds_s, queue_images.T) / temperature if queue_images.numel() > 0 else None  # Student 文本 VS 队列历史图像（纯负例）；返回 (batch_size, Q)，Q=队列当前有效特征数；队列空时为 None
        if neg_t2i is not None and neg_t2i.shape[1] > 0:  # 队列不为空时，在列方向（dim=1）拼接 batch 内矩阵与队列负样本矩阵
            logits_t2i = torch.cat([pos_t2i, neg_t2i], dim=1)                   # 合并后形状 (batch_size, batch_size+Q)：前 batch_size 列为 batch 内候选（含正例），后 Q 列为历史队列纯负样本
        else:                                                                      # 队列尚空（训练初始几步尚未积累历史特征）
            logits_t2i = pos_t2i                                                 # 仅使用 batch 内相似度矩阵；形状 (batch_size, batch_size)
        labels_t2i = torch.arange(batch_size, device=device)                     # 正样本索引标签；形状 (batch_size,)；labels_t2i[i]=i 表示第 i 条文本的正例是 logits_t2i 第 i 列（对角线）
        loss_t2i = F.cross_entropy(logits_t2i, labels_t2i)                      # 文本→图像 InfoNCE 对比损失（标量）；与 loss_i2t 对称，共同保证图文双向对齐
        # batch 内：另外 B-1 条文本是「同场考生」里的干扰项；
        # 队列里：Q 条历史文本是「从往届题库抽来的」干扰项；
        # 合在一起，每张图的负样本从 B-1 变成 (B-1)+Q，对比学习更难、也更有效
        contrastive_loss = (loss_i2t + loss_t2i) / 2                            # 双向对称对比损失的平均值（标量）；取平均保证 i2t 和 t2i 两个方向的对比学习权重相等

        # ---- 动量蒸馏：让 Student 的软分布接近 Teacher ----
        with torch.no_grad():  # 计算 Teacher 软标签不参与梯度，Teacher 特征已冻结，此处进一步保证无意外梯度
            teacher_logits = torch.matmul(image_embeds_m, text_embeds_m.T) / temperature  # Teacher 图像与 Teacher 文本的相似度矩阵（无梯度）；形状 (batch_size, batch_size)，[i,j] 为第 i 张 Teacher 图像与第 j 条 Teacher 文本的余弦相似度/温度
            teacher_probs = F.softmax(teacher_logits, dim=-1)                             # Teacher 软标签概率分布；形状 (batch_size, batch_size)；每行对 batch 内所有文本做 softmax，行 i 表示第 i 张图与各文本的匹配概率，非 one-hot，可缓解噪声配对
        student_logits = torch.matmul(image_embeds_s, text_embeds_m.detach().T) / temperature  # Student 图像与 Teacher 文本的相似度矩阵；形状 (batch_size, batch_size)；text_embeds_m.detach() 阻断 Teacher 侧梯度，只优化 Student 图像编码器
        student_log_probs = F.log_softmax(student_logits, dim=-1)                             # Student 的对数概率分布；形状 (batch_size, batch_size)；F.kl_div 要求第一个参数为对数概率（log_softmax 输出），第二个为目标概率
        distill_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')      # KL 散度蒸馏损失（标量）；reduction='batchmean' 对 batch 求平均；Student 分布越接近 Teacher 的软标签，损失越小

        total_loss = (1 - alpha) * contrastive_loss + alpha * distill_loss                   # 加权总损失（标量）；alpha 控制蒸馏强度，论文取 0.4；(1-alpha) 权重给 InfoNCE，alpha 权重给 KL 蒸馏

        return {
            'total_loss': total_loss,                   # 总损失（用于 backward）
            'contrastive_loss': contrastive_loss,       # 单纯 InfoNCE 对比损失
            'distill_loss': distill_loss,               # 动量蒸馏 KL 损失
            'queue_text_size': queue_texts.shape[0] if queue_texts.numel() > 0 else 0,     # 文本队列当前有效特征数
            'queue_image_size': queue_images.shape[0] if queue_images.numel() > 0 else 0,  # 图像队列当前有效特征数
        }

    def update_momentum_encoder(self):
        """
        每个 optimizer.step() 之后调用，用 EMA 同步 Teacher 参数。
        对图像编码器、文本编码器、图像投影头、文本投影头分别做 EMA 更新。
        """
        momentum_update(self.student.image_encoder, self.momentum_image_encoder, self.momentum)  # EMA 同步图像编码器
        momentum_update(self.student.text_encoder, self.momentum_text_encoder, self.momentum)    # EMA 同步文本编码器
        momentum_update(self.student.image_proj, self.momentum_image_proj, self.momentum)        # EMA 同步图像投影头
        momentum_update(self.student.text_proj, self.momentum_text_proj, self.momentum)          # EMA 同步文本投影头

    def forward(self, images, input_ids, attention_mask, task="contrastive_momentum", **kwargs):
        """
        统一前向入口，根据 task 参数分发任务。
        Args:
            images (List[PIL.Image.Image]): 原始图像列表，长度为 batch_size；由 ImageEncoder.forward() 内部的 ViTImageProcessor 统一完成 resize→normalize→转 Tensor 预处理
            input_ids (Tensor): 文本 token ID，形状 (batch_size, seq_len)
            attention_mask (Tensor): 文本 padding 掩码，形状 (batch_size, seq_len)
            task (str): 任务类型；"contrastive_momentum" 执行带动量蒸馏的 ITC，其余转发至 Student
            **kwargs: 透传给子任务的额外参数（如 alpha 蒸馏权重、itm labels 等）
        Returns:
            带动量蒸馏时返回损失字典（含 total_loss / contrastive_loss / distill_loss / queue_size）；
            其他任务返回 Student 模型的对应输出（损失标量或 logits）
        """
        if task == "contrastive_momentum":         # 带动量蒸馏的 ITC 任务分支
            return self.contrastive_forward_with_momentum(
                images, input_ids, attention_mask, alpha=kwargs.get('alpha', 0.4)  # 默认 alpha=0.4
            )  # 返回包含 total_loss / contrastive_loss / distill_loss 的损失字典
        return self.student(images, input_ids, attention_mask, task=task, **kwargs)  # 其余任务（ITM/MLM）直接转发给 Student 主模型

### 2.3 演示：动量蒸馏短程训练

运行 5 步训练，逐步观察负样本队列扩充过程及对比损失、蒸馏损失的变化趋势。

In [ ]:
def demo_momentum_distillation(queue_size=1024, train_steps=5, alpha=0.4):
    """
    短程训练演示：观察队列增长与各损失变化，验证动量蒸馏机制。
    可在普通 GPU/CPU 运行，无需 A100。
    Args:
        queue_size (int): 负样本队列大小，演示默认 1024
        train_steps (int): 训练步数，演示默认 5 步
        alpha (float): 蒸馏损失权重，演示默认 0.4
    """
    print("=" * 80)                              # 打印顶部分隔线，美化输出
    print("ALBEF 轻量版：动量蒸馏 + 负样本队列（教学演示）")  # 演示标题
    print("=" * 80)                              # 打印分隔线
    print(f"配置: queue_size={queue_size}, train_steps={train_steps}, alpha={alpha}, momentum=0.995")  # 打印关键超参数
    print("说明: 论文队列 65536 + 8×A100；此处为小规模演示，原理一致")  # 与论文规模对比说明
    print()                                      # 空行，分隔输出块

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 自动选择 GPU 或 CPU
    print(f"运行设备: {device}")                  # 打印当前使用的计算设备
    print()                                      # 空行

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # 加载 BERT 分词器
    texts = [                                    # 4 条英文演示句子，与 4 张随机图像一一对应
        "A cat sitting on a chair",
        "A dog running in the park",
        "A bird flying in the sky",
        "A fish swimming in water",
    ]
    tokenized = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=32)  # 批量分词；返回 dict，含 'input_ids' 形状 (4, seq_len) 和 'attention_mask' 形状 (4, seq_len)；padding=True 对齐到最长序列；max_length=32 为截断上限
    # 生成 4 张随机 RGB PIL 图像（演示用，无需真实图片）
    # ImageEncoder.forward() 内部调用 processor 完成 resize+归一化+设备转移，此处只需提供原始 PIL 图像列表
    images = [Image.fromarray(np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)) for _ in range(4)]
    input_ids = tokenized['input_ids'].to(device)                  # 文本 token ID 张量；形状 (4, seq_len)，seq_len ≤ 32；将数据从 CPU 搬到 device（CPU/GPU）
    attention_mask = tokenized['attention_mask'].to(device)        # 注意力掩码张量；形状 (4, seq_len)；1=有效 token，0=padding 位置（被 BERT 自注意力忽略）；搬到 device

    model = ALBEFWithMomentum(queue_size=queue_size, momentum=0.995).to(device)  # 实例化带动量的 ALBEF 并移到设备
    model.train()  # 训练模式：启用 dropout 等

    optimizer = torch.optim.AdamW(
        [p for p in model.student.parameters() if p.requires_grad],  # 只收集 Student 中 requires_grad=True 的参数（Teacher 已冻结，不会出现在此列表中）
        lr=1e-5,    # 学习率（float）；论文使用 1e-4~1e-5 量级，演示用 1e-5 防止随机权重下梯度爆炸
    )

    print("开始短程训练演示（仅 ITC + 动量蒸馏）...")  # 提示训练开始
    print("-" * 60)                              # 打印分隔线
    for step in range(1, train_steps + 1):       # 逐步训练，共 train_steps 步
        optimizer.zero_grad()  # 清空上一步梯度，防止梯度累积
        outputs = model(images, input_ids, attention_mask, task="contrastive_momentum", alpha=alpha)  # 带动量蒸馏的 ITC 前向；outputs 为 dict，含 total_loss/contrastive_loss/distill_loss（均为标量 Tensor）及 queue_text_size/queue_image_size（int，队列当前有效特征数）
        outputs['total_loss'].backward()  # 对总损失（标量）做反向传播，计算 Student 所有参数的梯度
        optimizer.step()                  # 用 AdamW 更新 Student 参数
        model.update_momentum_encoder()   # EMA 更新 Teacher 参数

        print(
            f"Step {step:02d} | total={outputs['total_loss'].item():.4f} "
            f"| contrastive={outputs['contrastive_loss'].item():.4f} "
            f"| distill={outputs['distill_loss'].item():.4f} "
            f"| text_queue={outputs['queue_text_size']} "   # 文本队列当前有效特征数
            f"| image_queue={outputs['queue_image_size']}"  # 图像队列当前有效特征数
        )

    print("-" * 60)                              # 打印结尾分隔线
    print("演示完成！")                           # 演示结束提示
    print("- 动量编码器: EMA 缓慢跟随 Student，提供稳定 Teacher 特征")  # 动量编码器功能说明
    print("- 负样本队列: 存放历史动量特征，扩充 ITC 负样本")           # 负样本队列功能说明
    print("- 动量蒸馏: KL 让 Student 学习 Teacher 的软概率分布")       # 动量蒸馏功能说明
    print("=" * 80)                              # 打印底部分隔线


# 运行轻量演示（约 1~3 分钟，普通 GPU/CPU 均可）
demo_momentum_distillation(queue_size=1024, train_steps=5, alpha=0.4)

## 补充说明

- `transformers` 库中**没有** ALBEF 官方实现，本 Notebook 为手写教学版。
- **Cell 1**：基础架构 + ITC / ITM / MLM 三任务演示（每行均有详细中文注释）。
- **Cell 2**：轻量版动量蒸馏 + 负样本队列（`queue_size=1024`，普通 GPU/CPU 可运行）。
- **建议学习顺序**：先通读 Cell 1 注释理解架构 → 运行演示 → 再学 Cell 2 动量机制。
